# 🔴 Solution: Sliding Window Attention

Reference solution — softmax attention with a band mask.

### Exercise contract
Support autograd through Q, K, and V, including when `d_v != d_k`. Compute `softmax(masked_scores, dim=-1) @ V`, with scores `Q @ K.transpose(-2, -1) / sqrt(d_k)`. The window is symmetric and includes both endpoints; `window_size` is a nonnegative integer. A dense band mask demonstrates the math but does **not** reduce the matrix multiplication cost.

In [ ]:
import torch
import math

In [ ]:
# ✅ SOLUTION

def sliding_window_attention(Q, K, V, window_size):
    d_k = K.size(-1)
    scores = torch.bmm(Q, K.transpose(1, 2)) / math.sqrt(d_k)
    S = Q.size(1)
    idx = torch.arange(S, device=Q.device)
    mask = (idx.unsqueeze(0) - idx.unsqueeze(1)).abs() > window_size
    scores = scores.masked_fill(mask.unsqueeze(0), float('-inf'))
    weights = torch.softmax(scores, dim=-1)
    return torch.bmm(weights, V)

In [ ]:
Q=torch.randn(1,6,8); K=torch.randn(1,6,8); V=torch.randn(1,6,8)
print('window=0==V?', torch.allclose(sliding_window_attention(Q,K,V,0), V, atol=1e-5))

In [ ]:
from torch_judge import check
check('sliding_window')